# Agent Tournament: Round-Robin Evaluation

Run all four agent types against each other in round-robin matchups.
Each pairing plays multiple games to produce statistically meaningful results.

**Agents:**
1. **RandomAgent** — baseline, random legal moves
2. **GreedyAgent** — hand-crafted heuristics (continent control, army ratios)
3. **ExpectiminimaxAgent** — adversarial search with dice probabilities
4. **MCTSAgent** — Monte Carlo Tree Search with UCB1

In [ ]:
from nb_setup import risk_ai_game
import random
import time

# Verify all agents import correctly
print("Available agents:")
print(f"  RandomAgent:         {risk_ai_game.RandomAgent}")
print(f"  AggressiveAgent:     {risk_ai_game.AggressiveAgent}")
print(f"  GreedyAgent:         {risk_ai_game.GreedyAgent}")
print(f"  ExpectiminimaxAgent: {risk_ai_game.ExpectiminimaxAgent}")
print(f"  MCTSAgent:           {risk_ai_game.MCTSAgent}")
print(f"  ExpectiminimaxAgent2: {risk_ai_game.ExpectiminimaxAgent2}")

## Quick Demo: One Game Per Matchup

First, let's run a single game for each agent pair to verify everything works.

In [ ]:
def make_agent(agent_cls, player_id, name):
    """Create an agent instance with sensible defaults."""
    if agent_cls == risk_ai_game.RandomAgent:
        return agent_cls(player_id, aggression=0.5, name=name)
    elif agent_cls == risk_ai_game.MCTSAgent:
        return agent_cls(player_id, num_simulations=20, max_rollout_turns=20, name=name)
    elif agent_cls == risk_ai_game.ExpectiminimaxAgent:
        return agent_cls(player_id, max_depth=2, top_k_moves=5, name=name)
    elif agent_cls == risk_ai_game.ExpectiminimaxAgent2:
        return agent_cls(player_id, max_depth=3, top_k=3, name=name)
    else:
        return agent_cls(player_id, name=name)


# If we need a lever to try to alleviate the first move bias, you can set this to 3.
INITIAL_ARMIES_PER_TERRITORY = 1

def make_initial_board_setup(armies_per_territory):
    def setup(game):
        territories = game.board.all_territories()
        random.shuffle(territories)
        for i, t in enumerate(territories):
            t.owner = i % game.num_players
            t.armies = armies_per_territory
        game.armies_to_deploy = game.get_reinforcements(game.current_player)

    return setup


board_setup = make_initial_board_setup(INITIAL_ARMIES_PER_TERRITORY)


def play_one_game_with_seat_trade(cls_a, name_a, cls_b, name_b, game_index, *, max_turns=500, verbose=False, random_seed=None, game_telemetry=None, initial_board_setup=None):
    if game_index % 2 == 0:
        agents = [make_agent(cls_a, 0, name_a), make_agent(cls_b, 1, name_b)]
    else:
        agents = [make_agent(cls_b, 0, name_b), make_agent(cls_a, 1, name_a)]
    winner = risk_ai_game.run_game(
        risk_ai_game.RiskAIGameOptions(
            agents=agents,
            max_turns=max_turns,
            verbose=verbose,
            random_seed=random_seed,
            game_telemetry=game_telemetry,
            initial_board_setup=initial_board_setup,
        )
    )
    if winner < 0:
        return "draw"
    if game_index % 2 == 0:
        return "a" if winner == 0 else "b"
    return "a" if winner == 1 else "b"


AGENT_TYPES = [
    ("Random", risk_ai_game.RandomAgent),
    ("Greedy", risk_ai_game.GreedyAgent),
    #("Expectiminimax", risk_ai_game.ExpectiminimaxAgent),
    ("Expectiminimax2", risk_ai_game.ExpectiminimaxAgent2),
    ("MCTS", risk_ai_game.MCTSAgent),
]

# Quick single-game test for each pair (trading seats every game)
for i, (name_a, cls_a) in enumerate(AGENT_TYPES):
    for j, (name_b, cls_b) in enumerate(AGENT_TYPES):
        if j <= i:
            continue
        t0 = time.time()
        outcome = play_one_game_with_seat_trade(
            cls_a, name_a, cls_b, name_b, 0, max_turns=500, verbose=False, initial_board_setup=board_setup
        )
        elapsed = time.time() - t0
        result = name_a if outcome == "a" else (name_b if outcome == "b" else "Draw")
        print(f"{name_a:18s} vs {name_b:18s} -> Winner: {result:18s} ({elapsed:.1f}s)")

## Full Round-Robin Tournament

Run N games for each matchup and collect win rates.

In [ ]:
import collections

GAMES_PER_MATCHUP = 20  # increase to 100+ for final report
MAX_TURNS = 500

# results[agent_name] = {wins, losses, draws, games}
results = {name: {"wins": 0, "losses": 0, "draws": 0, "games": 0}
           for name, _ in AGENT_TYPES}
matchup_results = {}

for i, (name_a, cls_a) in enumerate(AGENT_TYPES):
    for j, (name_b, cls_b) in enumerate(AGENT_TYPES):
        if j <= i:
            continue

        a_wins = 0
        b_wins = 0
        draws = 0
        t0 = time.time()

        for g in range(GAMES_PER_MATCHUP):
            # Alternate who goes first
            outcome = play_one_game_with_seat_trade(
                cls_a, name_a, cls_b, name_b, g, max_turns=MAX_TURNS, verbose=False, initial_board_setup=board_setup
            )
            if outcome == "a":
                a_wins += 1
            elif outcome == "b":
                b_wins += 1
            else:
                draws += 1

        elapsed = time.time() - t0
        key = f"{name_a} vs {name_b}"
        matchup_results[key] = {"a_wins": a_wins, "b_wins": b_wins, "draws": draws}
        results[name_a]["wins"] += a_wins
        results[name_a]["losses"] += b_wins
        results[name_a]["draws"] += draws
        results[name_a]["games"] += GAMES_PER_MATCHUP
        results[name_b]["wins"] += b_wins
        results[name_b]["losses"] += a_wins
        results[name_b]["draws"] += draws
        results[name_b]["games"] += GAMES_PER_MATCHUP

        pct_a = a_wins / GAMES_PER_MATCHUP * 100
        pct_b = b_wins / GAMES_PER_MATCHUP * 100
        print(f"{key:35s}  {name_a}: {a_wins:2d} ({pct_a:5.1f}%)  {name_b}: {b_wins:2d} ({pct_b:5.1f}%)  Draws: {draws}  [{elapsed:.1f}s]")

print("\n--- Overall Results ---")
for name, r in sorted(results.items(), key=lambda x: x[1]["wins"], reverse=True):
    win_pct = r['wins'] / max(1, r['games']) * 100
    print(f"{name:18s}  W:{r['wins']:3d}  L:{r['losses']:3d}  D:{r['draws']:3d}  Win%: {win_pct:.1f}%")

## Visualization: Win Rate Heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

agent_names = [name for name, _ in AGENT_TYPES]
n = len(agent_names)
matrix = np.full((n, n), np.nan)

for i, name_a in enumerate(agent_names):
    for j, name_b in enumerate(agent_names):
        if i == j:
            continue
        key = f"{name_a} vs {name_b}"
        key_rev = f"{name_b} vs {name_a}"
        if key in matchup_results:
            m = matchup_results[key]
            matrix[i][j] = m["a_wins"] / GAMES_PER_MATCHUP * 100
        elif key_rev in matchup_results:
            m = matchup_results[key_rev]
            matrix[i][j] = m["b_wins"] / GAMES_PER_MATCHUP * 100

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=100)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(agent_names, rotation=45, ha="right")
ax.set_yticklabels(agent_names)
ax.set_xlabel("Opponent")
ax.set_ylabel("Agent")
ax.set_title(f"Win Rate (%) — {GAMES_PER_MATCHUP} games per matchup")

for i in range(n):
    for j in range(n):
        if not np.isnan(matrix[i][j]):
            ax.text(j, i, f"{matrix[i][j]:.0f}%", ha="center", va="center",
                    fontsize=12, fontweight="bold")

plt.colorbar(im, label="Win %")
plt.tight_layout()
plt.show()

## Visualization: Overall Win Rates Bar Chart

In [ ]:
names = []
win_pcts = []
for name, r in sorted(results.items(), key=lambda x: x[1]["wins"], reverse=True):
    names.append(name)
    win_pcts.append(r["wins"] / max(1, r["games"]) * 100)

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, win_pcts, color=colors[:len(names)], edgecolor="black", alpha=0.85)
ax.set_ylabel("Win Rate (%)")
ax.set_title(f"Overall Win Rates — Round-Robin Tournament ({GAMES_PER_MATCHUP} games/matchup)")
ax.set_ylim(0, 100)
for bar, pct in zip(bars, win_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{pct:.1f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## Game Length Analysis by Matchup

In [ ]:
# Run a few games per matchup and track game lengths
SAMPLE_GAMES = 10
length_data = {}

for i, (name_a, cls_a) in enumerate(AGENT_TYPES):
    for j, (name_b, cls_b) in enumerate(AGENT_TYPES):
        if j <= i:
            continue
        key = f"{name_a} vs {name_b}"
        lengths = []
        for g in range(SAMPLE_GAMES):
            telemetry = risk_ai_game.TurnCountCollector()
            play_one_game_with_seat_trade(
                cls_a, name_a, cls_b, name_b, g,
                max_turns=500,
                verbose=False,
                game_telemetry=telemetry,
                initial_board_setup=board_setup,
            )
            lengths.append(telemetry.turns)
        length_data[key] = lengths
        avg = sum(lengths) / len(lengths)
        print(f"{key:35s}  avg turns: {avg:.0f}  min: {min(lengths)}  max: {max(lengths)}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(length_data.values(), labels=length_data.keys())
ax.set_ylabel("Turns")
ax.set_title("Game Length by Matchup")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()